In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostClassifier


In [30]:
df=pd.read_csv('train_apps.csv',sep=',')
df.head()
y=df['target_value']


In [31]:
df_feature=df.drop(columns=['target_value'],axis=1)


In [32]:
df_feature['cnt_deb_ul_ip_90'] = df_feature['cnt_deb_ul_ip_90'].fillna(df_feature['cnt_deb_ul_ip_90'].median())
df_feature['cnt_cred_loan_90_flag']=df_feature['cnt_cred_loan_90'].isna().astype(int)
df_feature['cnt_cred_loan_90'] = df_feature['cnt_cred_loan_90'].fillna(df_feature['cnt_cred_loan_90'].median())
df_feature['fl_hdb_bki_total_active_products'] = df_feature['fl_hdb_bki_total_active_products'].fillna(df_feature['fl_hdb_bki_total_active_products'].median())
df_feature['balance_rur_amt_30_min_flag']=df_feature['balance_rur_amt_30_min'].isna().astype(int)
df_feature['balance_rur_amt_30_min']=df_feature['balance_rur_amt_30_min'].fillna(0)
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] < -50,-9999)
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] > 50,9999)
df_feature['count_all_corp_dashboard_events'] = df_feature['count_all_corp_dashboard_events'].fillna(0)
df_feature=df_feature.drop(columns=['front_id','decision_day',
                                    'corp_credit_products','sum_deb_ul_90',
                                    'sum_deb_ul_30','cnt_deb_loan_90',
                                    'sum_deb_ul_90','cnt_deb_ul_ip_30',
                                    'loan_rev_max_start_non_fin','loan_rev_min_start_fin',
                                    'app_term_mean_360','overdraft_app_term_max_360','days_from_authperson_registration',
                                    'corp_list',
                                    'p75_time_spent_minutes','sum_deb_investment_90'],axis=1)

In [33]:
df_feature.dtypes

loan_amount_last                    float64
overdraft_limit_min                 float64
overdraft_limit_max                 float64
offered_rate                        float64
cb_rate                             float64
cnt_deb_ul_ip_90                    float64
balance_rur_amt_30_min              float64
cnt_cred_loan_90                    float64
fl_hdb_bki_total_active_products    float64
count_all_corp_dashboard_events     float64
db_group_last                        object
fl_adminarea                         object
cnt_cred_loan_90_flag                 int64
balance_rur_amt_30_min_flag           int64
dtype: object

In [34]:
# cat_features = ['db_group_last', 'fl_adminarea']

# df_feature = df_feature.copy()

# df_feature[cat_features] = (
#     df_feature[cat_features]
#     .astype(str)
#     .fillna("missing")
# )

# model = CatBoostClassifier(learning_rate=0.01,
#                            verbose=0,
#                            iterations=300,
#                            depth=3)

# param_grid = {
#     'learning_rate': [0.01, 0.05, 0.1],
#     'depth': [2, 3, 4],
#     'iterations': [300, 500, 800]
# }

# search = RandomizedSearchCV(
#     model,
#     param_distributions=param_grid,
#     cv=5,
#     scoring='roc_auc',
#     n_jobs=3,
#     verbose=2
# )

# model.fit(df_feature, y, cat_features=cat_features)

# y_prod_fit=model.predict_proba(df_feature)[:, 1]
# roc_auc_score(y,y_prod_fit)

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score


cat_features = ['db_group_last', 'fl_adminarea']


df_feature = df_feature.copy()
df_feature[cat_features] = (
    df_feature[cat_features]
    .astype(str)
    .fillna("missing")
)


model = CatBoostClassifier(
    verbose=0,
    random_seed=42  
)


param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'depth': [2, 3, 4, 6],
    'iterations': [300, 500, 800, 1000],
    'l2_leaf_reg': [1, 3, 5],  
    'border_count': [32, 64, 128] 
}


search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=20,  
    cv=5,
    scoring='roc_auc',
    n_jobs=3,
    verbose=2,
    random_state=42
)


search.fit(df_feature, y, cat_features=cat_features)


print("Лучшие параметры:", search.best_params_)
print("Лучший ROC-AUC (CV):", search.best_score_)


best_model = search.best_estimator_
y_pred = best_model.predict_proba(df_feature)[:, 1]
print("ROC-AUC на обучении:", roc_auc_score(y, y_pred))

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Лучшие параметры: {'learning_rate': 0.05, 'l2_leaf_reg': 3, 'iterations': 300, 'depth': 2, 'border_count': 64}
Лучший ROC-AUC (CV): 0.635663938842538
ROC-AUC на обучении: 0.802627058185239


In [36]:
del df_feature


In [37]:
df_feature=pd.read_csv('test_apps.csv',sep=',')
idx=df_feature['front_id']

In [38]:
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] < -50,-9999)
df_feature['offered_rate'] = df_feature['offered_rate'].mask(df_feature['offered_rate'] > 50,9999)
df_feature=df_feature.drop(columns=['front_id','decision_day'
                                   ],axis=1)
df_feature = df_feature.copy()

df_feature[cat_features] = (
    df_feature[cat_features]
    .astype(str)
    .fillna("missing")
)

In [39]:
# df_out = pd.DataFrame()
# df_out['front_id'] = idx
# df_out['target_value'] = model.predict_proba(df_feature)[:, 1]
# df_out.to_csv('submission.csv', index=False)